# Capítulo 3 · Variáveis e escalas de medida: caderno complementar
*Ciência de Dados e Inteligência Artificial em Saúde, Volume 1*

> **Este caderno acompanha o capítulo do livro; não o reproduz.** Definições, deduções e explicações estão no livro. Aqui você executa os cálculos do caso e confere os números. Todos os dados são **sintéticos** e os valores têm apenas finalidade didática.

A **Parte A** percorre o caso do capítulo, um dia sintético do registro de triagem de um pronto-socorro de adultos, pelas perguntas da equipe clínica; aparecem só as seções com cálculo (3.1 e 3.7). A **Parte B** traz a resolução escrita de todos os exercícios do capítulo (1 a 13; os enunciados estão no livro) e, nos exercícios cuja resolução precisa de cálculo (12 e 13), o código e o resultado. O código é Python 3 simples, só com a biblioteca padrão. Execute as células em ordem.

## Parte A. O caso

**Dados.** `data/triage.csv`: 200 linhas, uma por atendimento de 9 de março de 2026, e 12 colunas: `visit_id`, `arrived_at`, `seen_at` (vazio quando o paciente saiu antes de ser atendido), `arrival_route` (1 demanda espontânea, 2 ambulância, 3 transferência), `sex`, `age_years`, `triage_level` (1 mais urgente a 5 menos urgente), `pain` (nenhuma, leve, moderada, intensa), `temp_c` (vazia quando não registrada), `heart_rate`, `prior_visits` e `admitted`. `data/triage_check.csv`: os 12 primeiros atendimentos como foram digitados pela primeira vez, com três problemas de registro plantados para a etapa de validação. Cada célula traz o número da seção do livro a que pertence.

### 3.1 Introdução: os dados

In [1]:
# 1. Ler o registro: toda célula chega como texto, inclusive a idade
import csv
from datetime import datetime

with open("data/triage.csv", newline="") as stream:
    visits = list(csv.DictReader(stream))
print("O arquivo tem", len(visits), "registros (um por atendimento) e", len(visits[0]), "campos.")
print("Tipo de age_years ao ler:", type(visits[0]["age_years"]).__name__)

# 2. O identificador distingue os registros e encontra qualquer um deles; ele não mede nada
by_id = {}
for row in visits:
    by_id[row["visit_id"]] = row
print("visit_id distintos:", len(by_id))

O arquivo tem 200 registros (um por atendimento) e 12 campos.
Tipo de age_years ao ler: str
visit_id distintos: 200


### 3.7 O registro de triagem e seu dicionário

*O que significa a média de códigos do relatório, e como a escala de cada variável é registrada para que um programa possa aplicá-la?*

In [2]:
# 3. A média do relatório: o que o sistema fez com os códigos da via de chegada (duas numerações das mesmas categorias)
route_a = {"1": 1, "2": 2, "3": 3}
route_b = {"1": 3, "2": 1, "3": 2}
total_a = 0
total_b = 0
for row in visits:
    total_a = total_a + route_a[row["arrival_route"]]
    total_b = total_b + route_b[row["arrival_route"]]
print("Média da via com a numeração 1-2-3:", total_a / len(visits))
print("Média da via com a numeração 3-1-2:", total_b / len(visits))

# 4. O mesmo para o nível de triagem, que é ordinal: a ordem é mantida, o espaçamento muda
level_a = {"1": 1, "2": 2, "3": 3, "4": 4, "5": 5}
level_b = {"1": 1, "2": 2, "3": 4, "4": 8, "5": 16}
total_a = 0
total_b = 0
for row in visits:
    total_a = total_a + level_a[row["triage_level"]]
    total_b = total_b + level_b[row["triage_level"]]
print("Média do nível com o espaçamento 1-2-3-4-5:", total_a / len(visits))
print("Média do nível com o espaçamento 1-2-4-8-16:", total_b / len(visits))

Média da via com a numeração 1-2-3: 1.29
Média da via com a numeração 3-1-2: 2.615
Média do nível com o espaçamento 1-2-3-4-5: 3.345
Média do nível com o espaçamento 1-2-4-8-16: 6.035


In [3]:
# 5. Variáveis derivadas: tempo até ser atendido (minutos) e febre; um atendimento sem horário de atendimento é de quem saiu antes de ser atendido
for row in visits:
    row["wait_min"] = None
    if row["seen_at"] != "":
        arrived = datetime.fromisoformat(row["arrived_at"])
        seen = datetime.fromisoformat(row["seen_at"])
        row["wait_min"] = (seen - arrived).total_seconds() / 60
    row["fever"] = None
    if row["temp_c"] != "":
        if float(row["temp_c"]) >= 38.0:
            row["fever"] = "yes"
        else:
            row["fever"] = "no"
print("Tempo até ser atendido, V0001:", by_id["V0001"]["wait_min"], "minutos; V0002:", by_id["V0002"]["wait_min"], "minutos")
left = []
for row in visits:
    if row["wait_min"] is None:
        left.append(row["visit_id"])
print("Saíram antes de ser atendidos:", len(left), "atendimentos", left)
for visit_id in ["V0074", "V0026", "V0174"]:
    print("Atendimento", visit_id, "- temperatura", by_id[visit_id]["temp_c"], "- febre:", by_id[visit_id]["fever"])

Tempo até ser atendido, V0001: 87.0 minutos; V0002: 25.0 minutos
Saíram antes de ser atendidos: 3 atendimentos ['V0025', 'V0050', 'V0093']
Atendimento V0074 - temperatura 37.6 - febre: no
Atendimento V0026 - temperatura 38.2 - febre: yes
Atendimento V0174 - temperatura 40.0 - febre: yes


In [4]:
# 6. A escala de cada variável decide o resumo: contagens para categorias
SCALE = {"visit_id": "identificador", "arrival_route": "nominal", "sex": "nominal",
         "admitted": "nominal", "fever": "nominal", "triage_level": "ordinal",
         "pain": "ordinal", "arrived_at": "intervalar", "seen_at": "intervalar",
         "temp_c": "intervalar", "age_years": "razão", "heart_rate": "razão",
         "prior_visits": "razão", "wait_min": "razão"}
for scale in ["identificador", "nominal", "ordinal", "intervalar", "razão"]:
    names = []
    for name in SCALE:
        if SCALE[name] == scale:
            names.append(name)
    print(scale + ":", ", ".join(names))

# As contagens das variáveis nominais e ordinais, na ordem da escala (valores faltantes são contados à parte)
ORDER = {"arrival_route": ["1", "2", "3"], "sex": ["F", "M"],
         "admitted": ["yes", "no"], "fever": ["yes", "no"],
         "triage_level": ["1", "2", "3", "4", "5"],
         "pain": ["none", "mild", "moderate", "severe"]}
for variable in SCALE:
    if SCALE[variable] in ("nominal", "ordinal"):
        counts = {}
        for value in ORDER[variable]:
            counts[value] = 0
        counts["faltante"] = 0
        for row in visits:
            value = row[variable]
            if value is None or value == "":
                value = "faltante"
            counts[value] = counts[value] + 1
        shown = {}
        for value in counts:
            if counts[value] > 0:
                shown[value] = counts[value]
        print("Contagens de", variable, "(" + SCALE[variable] + "):", shown)

identificador: visit_id
nominal: arrival_route, sex, admitted, fever
ordinal: triage_level, pain
intervalar: arrived_at, seen_at, temp_c
razão: age_years, heart_rate, prior_visits, wait_min
Contagens de arrival_route (nominal): {'1': 155, '2': 32, '3': 13}
Contagens de sex (nominal): {'F': 102, 'M': 98}
Contagens de admitted (nominal): {'yes': 54, 'no': 146}
Contagens de fever (nominal): {'yes': 44, 'no': 152, 'faltante': 4}
Contagens de triage_level (ordinal): {'1': 5, '2': 23, '3': 85, '4': 72, '5': 15}
Contagens de pain (ordinal): {'none': 38, 'mild': 80, 'moderate': 51, 'severe': 31}


In [5]:
# 7. Intervalar e razão: as mesmas três temperaturas em três unidades e as mesmas duas idades em duas unidades
celsius = [37.0, 38.0, 39.0]
fahrenheit = []
kelvin = []
for c in celsius:
    fahrenheit.append(round(1.8 * c + 32, 1))
    kelvin.append(round(c + 273.15, 2))
print("Celsius", celsius, "- razão do último para o primeiro:", round(celsius[2] / celsius[0], 3))
print("Fahrenheit", fahrenheit, "- razão do último para o primeiro:", round(fahrenheit[2] / fahrenheit[0], 3))
print("Kelvin", kelvin, "- razão do último para o primeiro:", round(kelvin[2] / kelvin[0], 3))
print("Razão de idades, 80 anos contra 40 anos:", 80 / 40, "; em meses:", (80 * 12) / (40 * 12))

Celsius [37.0, 38.0, 39.0] - razão do último para o primeiro: 1.054
Fahrenheit [98.6, 100.4, 102.2] - razão do último para o primeiro: 1.037
Kelvin [310.15, 311.15, 312.15] - razão do último para o primeiro: 1.006
Razão de idades, 80 anos contra 40 anos: 2.0 ; em meses: 2.0


In [6]:
# 8. As regras do dicionário, aplicadas a um arquivo pequeno digitado à mão
rules = {
    "age_years": {"converter": int, "minimo": 18, "maximo": 110},
    "triage_level": {"converter": int, "minimo": 1, "maximo": 5},
    "temp_c": {"converter": float, "minimo": 30.0, "maximo": 43.0},
}
with open("data/triage_check.csv", newline="") as stream:
    check = list(csv.DictReader(stream))

validated = []
issues = []
for row in check:
    clean = {"visit_id": row["visit_id"]}
    for name in rules:
        rule = rules[name]
        text = row[name]
        clean[name] = None
        if text == "":
            print(row["visit_id"], name, "faltante")
        else:
            value = rule["converter"](text)
            if rule["minimo"] <= value <= rule["maximo"]:
                clean[name] = value
            else:
                issues.append((row["visit_id"], name, text))
                print(row["visit_id"], name, "não permitido:", value)
    validated.append(clean)
print("Para revisão:", issues)
checked = {}
for record in validated:
    checked[record["visit_id"]] = record
print("Nível de triagem de V0004 após a validação:", checked["V0004"]["triage_level"])

V0004 triage_level não permitido: 6
V0007 temp_c faltante
V0008 age_years não permitido: 430
V0010 temp_c faltante
Para revisão: [('V0004', 'triage_level', '6'), ('V0008', 'age_years', '430')]
Nível de triagem de V0004 após a validação: None


In [7]:
# 9. Os resumos que a próxima seção relata
n = len(visits)
urgent = 0
ambulance = 0
admitted = 0
fever_yes = 0
fever_no = 0
for row in visits:
    if row["triage_level"] == "1" or row["triage_level"] == "2":
        urgent = urgent + 1
    if row["arrival_route"] == "2":
        ambulance = ambulance + 1
    if row["admitted"] == "yes":
        admitted = admitted + 1
    if row["fever"] == "yes":
        fever_yes = fever_yes + 1
    if row["fever"] == "no":
        fever_no = fever_no + 1
known = fever_yes + fever_no
print("Níveis de triagem 1 e 2:", urgent, "de", n, "atendimentos =", round(100 * urgent / n, 1), "%")
print("Ambulância direta:", ambulance, "de", n, "atendimentos =", round(100 * ambulance / n, 1), "%")
print("Internados:", admitted, "de", n, "atendimentos =", round(100 * admitted / n, 1), "%")
print("Febre:", fever_yes, "de", known, "atendimentos com temperatura registrada =", round(100 * fever_yes / known, 1), "%")

Níveis de triagem 1 e 2: 28 de 200 atendimentos = 14.0 %
Ambulância direta: 32 de 200 atendimentos = 16.0 %
Internados: 54 de 200 atendimentos = 27.0 %
Febre: 44 de 196 atendimentos com temperatura registrada = 22.4 %


## Parte B. Soluções dos exercícios

### 3.9 Exercício 1. Um registro de insuficiência cardíaca

A unidade de observação é um paciente inscrito; cada linha é um paciente. As
variáveis são sexo, classe funcional, fração de ejeção, número de internações e
data de inscrição. Registro é um identificador: distingue as linhas e não mede
nada. Ambulatório tem o valor Cardiologia nas quatro linhas, portanto é uma
constante neste conjunto de dados e não distingue um paciente de outro.

### 3.9 Exercício 2. Cinco colunas em um laboratório

O número de acesso é um identificador. Tipo de espécime, grau de hemólise e
rejeição são variáveis qualitativas: seus valores são categorias. A concentração
de potássio é uma variável quantitativa: resulta de uma medição.

### 3.9 Exercício 3. Contando tipos de espécime

O denominador são os 500 espécimes recebidos naquela semana. O sangue responde por
$320/500=64\%$, a urina por $140/500=28\%$ e os outros espécimes por $40/500=8\%$.
As três proporções somam 100%. O tipo de espécime é nominal, então os códigos 1, 2
e 3 são nomes. A média deles, $(320\times1+140\times2+40\times3)/500=1{,}44$,
mudaria se os códigos fossem reatribuídos com os mesmos espécimes. Ela descreve a
numeração escolhida pelo sistema.

### 3.9 Exercício 4. Classe funcional em uma coorte

A classe funcional é qualitativa em escala ordinal: as quatro classes têm uma
ordem, e as distâncias entre elas não são definidas. As classes III ou IV somam
$45+15=60$ pacientes, ou $60/150=40\%$. A escala não permite afirmar que um
paciente da classe IV é duas vezes mais limitado que um da classe II, nem que o
passo de I para II equivale ao passo de III para IV.

### 3.9 Exercício 5. Duas codificações da dor

Com os códigos 0, 1, 2, 3, a enfermaria A dá $(0+0+3+3)/4=1{,}5$ e a enfermaria B
dá $(1+2+2+2)/4=1{,}75$, então a enfermaria B parece ter mais dor. Com os códigos
0, 1, 2, 10, a enfermaria A dá $(0+0+10+10)/4=5{,}0$ e a B dá de novo 1,75, então a
A parece ter mais dor. As duas codificações respeitam a ordem das categorias, e a
comparação se inverte. Contagens e proporções não dependem da codificação: na
enfermaria A, dois dos quatro pacientes tiveram dor intensa e dois não tiveram
nenhuma; na B, nenhum teve dor intensa e todos os quatro tiveram alguma dor.

### 3.9 Exercício 6. Afirmações com e sem significado

(a) Tem significado: a concentração está em escala de razão, e zero significa ausência de potássio.
(b) Não tem significado: graus Celsius formam uma escala intervalar com zero arbitrário.
(c) Tem significado: as datas estão em escala intervalar, e a afirmação é uma diferença.
(d) Tem significado: uma duração está em escala de razão.
(e) Não tem significado: o nível de triagem é ordinal, e seus dígitos são rótulos ordenados.

### 3.9 Exercício 7. Diferenças e quocientes em duas unidades

Em Celsius, o aumento do primeiro ao terceiro paciente é $39{,}5-36{,}5=3{,}0$ e
do primeiro ao segundo $37{,}5-36{,}5=1{,}0$, com quociente 3. Em Fahrenheit os
valores são 97,7, 99,5 e 103,1; os aumentos são 5,4 e 1,8, e o quociente é de novo
3. O quociente entre o terceiro e o primeiro valor é $39{,}5/36{,}5\approx1{,}082$
em Celsius e $103{,}1/97{,}7\approx1{,}055$ em Fahrenheit. Isto decorre dos zeros arbitrários das escalas Celsius e Fahrenheit: as razões entre diferenças se preservam, e as razões entre valores não.

### 3.9 Exercício 8. Discreto ou contínuo

O número de medicamentos e o número de quedas são discretos: resultam de
contagem. Peso, pressão arterial sistólica e creatinina sérica são contínuos:
resultam de medição, e um instrumento mais fino distinguiria valores intermediários.
Uma pressão arterial registrada ao múltiplo de 2 mmHg mais próximo é um
número inteiro por construção, e a variável continua sendo contínua. A regra de registro limita os
valores armazenados; não altera a natureza da quantidade.

### 3.9 Exercício 9. Índice de massa corporal e obesidade

O quadrado da altura é $1{,}80^2=3{,}24$. Os índices são $96{,}9/3{,}24\approx29{,}9$,
$97{,}5/3{,}24\approx30{,}1$ e $145{,}8/3{,}24=45{,}0$. Pela definição da clínica, o primeiro
paciente não é obeso e os outros dois são. Os dois primeiros pacientes são os mais
parecidos no índice, com diferença de 0,2, e ficam em categorias diferentes. O
segundo e o terceiro são parecidos na categoria e diferem em 14,9. O índice é uma
variável quantitativa derivada em escala de razão. A situação de obesidade é uma
variável binária derivada em escala nominal.

### 3.9 Exercício 10. Uma idade registrada em grupos

O registro não pode fornecer a proporção com 70 anos ou mais. O grupo de 65 e mais
contém esses pacientes junto com os de 65 a 69 anos, e nada no registro os separa.
Deveria ter sido registrada a idade em anos, ou a data de nascimento junto com a data do registro; qualquer
agrupamento de idades em anos poderia então ter sido derivado. A categorização vai do registro mais fino
para o mais grosseiro e não pode ser invertida.

### 3.9 Exercício 11. Um dicionário de dados para o registro

| Variável | Tipo | Escala | Valores ou unidade |
|---|---|---|---|
| Registro | identificador |  | um por paciente |
| Ambulatório | qualitativa | nominal | constante neste arquivo (Cardiologia em todas as linhas) |
| Sexo | qualitativa, binária | nominal | F, M |
| Classe | qualitativa | ordinal | I, II, III, IV |
| FE | quantitativa, contínua | razão | percentual, 0 a 100 |
| Internações | quantitativa, discreta | razão | 0, 1, 2, ... no ano anterior |
| Inscrição | quantitativa | intervalar | data de calendário |

### 3.9 Exercício 12. Dor e atendimentos anteriores no registro da triagem

As contagens na ordem da escala são 38, 80, 51 e 31 para nenhuma, leve,
moderada e intensa. Dor pelo menos moderada foi registrada em $51+31=82$ dos 200
atendimentos, ou 41%. Após a conversão com `int`, 29 atendimentos foram de
pacientes com três ou mais atendimentos nos 12 meses anteriores, ou 14,5%.

In [8]:
# 1. Contar as categorias de dor na ordem da escala e os atendimentos de pacientes com três ou mais atendimentos anteriores
order = ["none", "mild", "moderate", "severe"]
pain_counts = {}
for category in order:
    pain_counts[category] = 0
frequent = 0
for row in visits:
    pain_counts[row["pain"]] = pain_counts[row["pain"]] + 1
    if int(row["prior_visits"]) >= 3:
        frequent = frequent + 1
print("Contagens de dor:", pain_counts)

# 2. As duas proporções
at_least_moderate = pain_counts["moderate"] + pain_counts["severe"]
print("Dor pelo menos moderada:", at_least_moderate, "de", len(visits), "atendimentos =", round(100 * at_least_moderate / len(visits), 1), "%")
print("Três ou mais atendimentos nos 12 meses anteriores:", frequent, "de", len(visits), "atendimentos =", round(100 * frequent / len(visits), 1), "%")

Contagens de dor: {'none': 38, 'mild': 80, 'moderate': 51, 'severe': 31}
Dor pelo menos moderada: 82 de 200 atendimentos = 41.0 %
Três ou mais atendimentos nos 12 meses anteriores: 29 de 200 atendimentos = 14.5 %


### 3.9 Exercício 13. Uma regra para a frequência cardíaca

Para este exercício, escolha 30 e 220 batimentos por minuto como limites que
acionam revisão, e não como fronteiras das frequências cardíacas possíveis. A regra é
`{"converter": int, "minimo": 30, "maximo": 220}`, aplicada registro a registro: cada frequência cardíaca é convertida
e comparada com os dois limites. Com esses limites, nenhuma
frequência cardíaca dos 200 registros está faltante e nenhuma é sinalizada. Um valor
dentro dos limites ainda pode estar errado: 84 digitado em vez de 48 passa na
verificação de faixa. Um valor fora deles exige investigação, e não correção
automática nem exclusão. A verificação exige comparação com o registro de origem. Os limites são escolhas didáticas,
largos o bastante para que nenhuma frequência cardíaca plausível de adulto seja sinalizada. O laço abaixo aplica a regra aos 200 registros.

In [9]:
# 1. A regra para a frequência cardíaca: limites que acionam revisão (limites de plausibilidade, não as fronteiras das frequências possíveis)
rule = {"converter": int, "minimo": 30, "maximo": 220}

# 2. Aplicá-la aos 200 registros
missing = 0
flagged = []
for row in visits:
    text = row["heart_rate"]
    if text == "":
        missing = missing + 1
    else:
        value = rule["converter"](text)
        if not (rule["minimo"] <= value <= rule["maximo"]):
            flagged.append(row["visit_id"])
print("Frequências cardíacas faltantes:", missing)
print("Frequências cardíacas não permitidas:", len(flagged))

Frequências cardíacas faltantes: 0
Frequências cardíacas não permitidas: 0


---
*Dados sintéticos (`generate_data.py`, semente 3). Valores apenas didáticos; nada aqui é estimativa para um departamento real.*